# LoCED × MaxText v0.2.5 — 500M Colab TPU smoke

This notebook runs a **~500.2M-parameter LoCED** scale smoke on a single
Google Colab TPU (**v5e-1, 16 GB**):

- **4 physical layers, 8 visits** — the same encode/decode schedule as the reference;
- **~500.23M parameters** — `d=1024`, 8 experts, expert width 3840, vocab 32k;
- builds the model and runs **two MaxText Muon updates** on one synthetic batch;
- reports parameter count, first/second step time, loss, optimizer step, and peak memory.

### Colab checklist

1. **Runtime → Change runtime type → TPU**, then save.
2. Run the cells in order. The install takes roughly 5–15 minutes.
3. Keep the tab open for the whole run (~10–20 minutes).

This is a **scale smoke**, not the 15-test tiny parity suite. The exact parity
suite, sharding checks, and Orbax checkpoint round-trip stay in
`LoCED_MaxText_Colab_TPU.ipynb`. This notebook deliberately does not write a
checkpoint and does not claim 500M convergence.

### What “pass” means here

- parameter count is in the **495M–505M** band (expected 500,229,136);
- backend is `tpu` with one device;
- two optimizer steps complete, loss is finite, and a parameter leaf changes;
- peak memory fits the 16 GB v5e-1 slice. If it OOMs, use a larger TPU.


In [ ]:
# ============================================================================
# LoCED x MaxText v0.2.5 - 500M Colab TPU smoke
# ============================================================================
import json
import os
import shutil
import subprocess
import sys
import textwrap
import time
from pathlib import Path

# ---------------------------------------------------------------------------
# User knobs
# ---------------------------------------------------------------------------
ACCELERATOR = os.environ.get("LOCED_ACCELERATOR", "tpu")  # TPU validation
MAXTEXT_REF = "maxtext-v0.2.5"
MAXTEXT_REPO = "https://github.com/AI-Hypercomputer/maxtext.git"
MAXTEXT_EXTRAS = "cuda12" if ACCELERATOR == "gpu" else "tpu"
MODEL_CONFIG = "loced/maxtext/configs/loced_500m_tpu.yml"

# The notebook clones this repo if LoCED is not already present.
LOCED_REPO_URL = os.environ.get(
    "LOCED_REPO_URL", "https://github.com/offx-zinth/loced.git"
)
LOCED_REPO_REF = os.environ.get("LOCED_REPO_REF", "master")

WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
WORK.mkdir(parents=True, exist_ok=True)
MAXTEXT_DIR = WORK / "maxtext-v0.2.5"
VENV_DIR = WORK / "loced-venv"
PY = VENV_DIR / "bin" / "python"
SCRIPTS = WORK / "loced_scripts"
RUNS = WORK / "loced_runs"
CACHE = WORK / "jax_cache"
REPORT_500M = WORK / "colab_tpu_500m_report.json"
for _p in (SCRIPTS, RUNS, CACHE):
    _p.mkdir(parents=True, exist_ok=True)


def _loced_parent(root: Path):
    # Return the directory to put on PYTHONPATH for `import loced`.
    if (root / "loced" / "maxtext" / "train.py").is_file():
        return root
    # offx-zinth/loced is the package contents at the repo root.
    if root.name == "loced" and (root / "maxtext" / "train.py").is_file():
        return root.parent
    return None


def _find_loced_root():
    candidates = []
    if os.environ.get("LOCED_ROOT"):
        candidates.append(Path(os.environ["LOCED_ROOT"]))
    candidates += [
        WORK,
        WORK / "loced",
        Path.cwd(),
        Path.cwd() / "loced",
        Path("/kaggle/input"),
    ]
    for root in candidates:
        found = _loced_parent(root)
        if found is not None:
            return found
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        for p in kaggle_input.glob("**/maxtext/train.py"):
            repo = p.parents[1]
            if repo.name == "loced" or (
                (repo / "attention.py").is_file()
                and (repo / "config.py").is_file()
            ):
                return repo.parent
        for p in kaggle_input.glob("**/loced/maxtext/train.py"):
            return p.parents[2]
    return None


# Keep a reused Kaggle working directory in sync with the branch under test.
_target = WORK / "loced"
if LOCED_REPO_URL and (_target / ".git").is_dir():
    _env = os.environ.copy()
    _env["GIT_TERMINAL_PROMPT"] = "0"
    print(f"updating existing checkout {_target} ({LOCED_REPO_REF})")
    subprocess.run(
        ["git", "-C", str(_target), "fetch", "--depth", "1",
         "origin", LOCED_REPO_REF],
        check=True, env=_env,
    )
    subprocess.run(
        ["git", "-C", str(_target), "reset", "--hard", "FETCH_HEAD"],
        check=True, env=_env,
    )

LOCED_ROOT = _find_loced_root()
if LOCED_ROOT is None and LOCED_REPO_URL:
    target = WORK / "loced"
    if not (target / ".git").is_dir():
        if target.is_dir() and any(target.iterdir()):
            print(f"{target} exists without .git; treating it as the source")
        else:
            env = os.environ.copy()
            env["GIT_TERMINAL_PROMPT"] = "0"
            print(f"cloning {LOCED_REPO_URL} ({LOCED_REPO_REF}) -> {target}")
            subprocess.run(
                ["git", "clone", "--depth", "1", "--branch", LOCED_REPO_REF,
                 LOCED_REPO_URL, str(target)],
                check=True, env=env,
            )
    LOCED_ROOT = _find_loced_root()

print(f"ACCELERATOR : {ACCELERATOR}")
print(f"WORK        : {WORK}")
print(f"LOCED_ROOT  : {LOCED_ROOT}")
print(f"MAXTEXT_DIR : {MAXTEXT_DIR}")
print(f"VENV        : {VENV_DIR}")
if LOCED_ROOT is None:
    raise RuntimeError(textwrap.dedent('''
        LoCED source not found.
        - Upload the repository folder (containing `loced/`) to
          /kaggle/working/loced, or as a Kaggle Dataset under
          /kaggle/input/...
        - Or set LOCED_REPO_URL (and LOCED_REPO_REF) in this cell to a branch
          that contains `loced/`.
    '''))
print("nvidia-smi:")
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi", "-L"], check=False)
else:
    print("nvidia-smi not found (CPU-only notebook kernel?)")

## 1. Python 3.12 + MaxText v0.2.5 (TPU)

MaxText v0.2.5 requires Python >= 3.12, so the notebook creates its own venv
with `uv` even if the Kaggle kernel is older. The install uses MaxText's tested
dependency set:

```bash
uv pip install -e .[tpu] --resolution=lowest
```

Set `LOCED_SKIP_INSTALL=1` only when re-running against an already prepared
venv.

In [ ]:
SKIP_INSTALL = os.environ.get("LOCED_SKIP_INSTALL", "0") == "1"

if SKIP_INSTALL:
    print("LOCED_SKIP_INSTALL=1: using existing venv/MaxText checkout")
else:
    if shutil.which("uv") is None:
        print("installing uv ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    UV = shutil.which("uv")
    if UV is None:
        raise RuntimeError("uv is not on PATH after pip install")

    if not PY.exists():
        print("creating Python 3.12 venv ...")
        subprocess.run(
            [UV, "venv", "--python", "3.12", "--seed", str(VENV_DIR)],
            check=True,
        )
    print(subprocess.check_output([str(PY), "--version"], text=True).strip())

    if not (MAXTEXT_DIR / "pyproject.toml").is_file():
        print(f"cloning MaxText {MAXTEXT_REF} ...")
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", MAXTEXT_REF,
             MAXTEXT_REPO, str(MAXTEXT_DIR)],
            check=True,
        )
    else:
        print("MaxText checkout already present")

    print(f"installing maxtext[{MAXTEXT_EXTRAS}] (resolution=lowest) ...")
    subprocess.run(
        [UV, "pip", "install", "--python", str(PY), "--link-mode=copy",
         "-e", f".[{MAXTEXT_EXTRAS}]", "--resolution=lowest"],
        cwd=str(MAXTEXT_DIR), check=True,
    )
    subprocess.run(
        [UV, "pip", "install", "--python", str(PY), "--link-mode=copy",
         "pytest", "pytest-timeout"],
        check=True,
    )
    print("dependency check:")
    subprocess.run([UV, "pip", "check", "--python", str(PY)], check=False)

print(subprocess.check_output(
    [str(PY), "-c",
     "import sys, maxtext; print(sys.version); print(maxtext.__file__)"],
    text=True,
))

## 2. TPU + real-MaxText sanity check

This is deliberately strict: the run fails if JAX is not on the TPU, or if
MaxText's heavy utilities cannot be imported. A CPU-stub fallback would make the
500M result meaningless.

In [ ]:
BASE_ENV = os.environ.copy()
BASE_ENV.update({
    "PYTHONPATH": f"{MAXTEXT_DIR / 'src'}:{LOCED_ROOT}",
    "JAX_COMPILATION_CACHE_DIR": str(CACHE),
    "XLA_PYTHON_CLIENT_MEM_FRACTION": "0.90",
    "XLA_PYTHON_CLIENT_PREALLOCATE": "true",
    "TOKENIZERS_PARALLELISM": "false",
    "HF_HUB_DISABLE_PROGRESS_BARS": "1",
    "PYTHONDONTWRITEBYTECODE": "1",
    "TF_CPP_MIN_LOG_LEVEL": "2",
})

# jax-cuda12 loads the pip-installed NVIDIA libraries with ctypes.  In a venv
# they are under site-packages/nvidia/*/lib, which is not on the default loader
# path, so cuSPARSE/cuDNN can otherwise be reported as "not found".
_nvidia_code = (
    "import site; from pathlib import Path; "
    "print(':'.join(str(p) for r in site.getsitepackages() "
    "for p in sorted((Path(r) / 'nvidia').glob('*/lib')) if p.is_dir()))"
)
_nvidia_libs = subprocess.check_output([str(PY), "-c", _nvidia_code], text=True).strip()
if _nvidia_libs:
    BASE_ENV["LD_LIBRARY_PATH"] = ":".join(
        x for x in (_nvidia_libs, os.environ.get("LD_LIBRARY_PATH", "")) if x
    )
print("NVIDIA lib dirs:", len(_nvidia_libs.split(":")) if _nvidia_libs else 0)


def run_script(name, code, timeout=3600, extra_env=None, check=True, cwd=None):
    path = SCRIPTS / name
    path.write_text(textwrap.dedent(code))
    env = BASE_ENV.copy()
    if extra_env:
        env.update({str(k): str(v) for k, v in extra_env.items()})
    t0 = time.time()
    proc = subprocess.run(
        [str(PY), str(path)],
        cwd=str(cwd or LOCED_ROOT),
        env=env,
        text=True,
        capture_output=True,
        timeout=timeout,
    )
    elapsed = time.time() - t0
    print(proc.stdout)
    if proc.stderr.strip():
        print("[stderr]\n" + proc.stderr[-12000:])
    print(f"[{name}] exit={proc.returncode} elapsed={elapsed:.1f}s")
    if check and proc.returncode != 0:
        raise RuntimeError(f"{name} failed (exit {proc.returncode})")
    return proc


gpu_check = r'''
import importlib.metadata
import os
import shutil
import subprocess
import sys

import jax
import maxtext

print("python  :", sys.version.split()[0])
print("jax     :", jax.__version__)
print("maxtext :", importlib.metadata.version("maxtext"))
print("backend :", jax.default_backend())
print("devices :", jax.devices())

expected = os.environ["LOCED_EXPECTED_BACKEND"]
assert jax.default_backend() == expected, (
    f"Expected {expected}, got {jax.default_backend()}. "
    "Set the Colab TPU runtime and rerun this cell."
)
assert len(jax.devices()) >= 1
for d in jax.devices():
    stats = d.memory_stats() or {}
    print(f"  {d}: {d.device_kind} bytes_limit={stats.get('bytes_limit')}")

# The GPU/TPU run must exercise the real MaxText utilities, not CPU stubs.
import maxtext.utils.maxtext_utils  # noqa: F401
import maxtext.utils.model_creation_utils  # noqa: F401
print("real MaxText utilities: OK")
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
'''
run_script(
    "gpu_check.py", gpu_check,
    extra_env={"LOCED_EXPECTED_BACKEND": ACCELERATOR},
)

## 3. ~500M parameter scale smoke

Builds `configs/models/loced-500m.yml` through MaxText's real `pyconfig`, then
runs two MaxText Muon updates with a batch of 1 and sequence length 64:

- `d=1024`, 4 layers, 8 visits, 8 routed experts + 1 shared expert;
- **500,229,136 parameters** by construction;
- float32 reference compute (the current LoCED core), so this is the memory
  worst case; it is not a bf16 throughput benchmark.


In [ ]:
bench_code = r'''
import json
import os

from loced.maxtext.benchmark_500m import main

report = main()
print(json.dumps(report, indent=2))
'''
run_script(
    "benchmark_500m.py",
    bench_code,
    timeout=7200,
    extra_env={
        "LOCED_500M_CONFIG": MODEL_CONFIG,
        "LOCED_500M_REPORT": str(REPORT_500M),
        "LOCED_EXPECT_BACKEND": ACCELERATOR,
    },
)


## 4. Summary

If all cells completed, the ~500M LoCED model built and trained for two
MaxText Muon steps on a single TPU chip. The raw report is
`WORK/colab_tpu_500m_report.json`.

This is intentionally a scale/plumbing smoke: the 15-test exact parity suite
and the Orbax checkpoint round-trip remain in `LoCED_MaxText_Colab_TPU.ipynb`.


In [ ]:
import json
from pathlib import Path

p = Path(REPORT_500M)
if p.is_file():
    report = json.loads(p.read_text())
    print(json.dumps(report, indent=2))
    print("500M TPU smoke complete.")
else:
    print("missing report:", p)
